# 🛡️ Интеллектуальная система формирования требований ИБ

ИИ-агент, который по **архитектурной схеме** программной системы и её параметрам
(класс критичности, наличие персональных данных) автоматически формирует перечень
применимых **требований информационной безопасности**.

**Технологии:** мультимодальная LLM (YandexGPT), OCR (Yandex Vision), few-shot обучение,
трёхэтапный отбор требований с ранжированием по TF-IDF, оценка качества по метрикам
Precision / Recall / F1.

> Курсовой проект, Уфимский университет науки и технологий, 2026.
>
> ⚠️ Ноутбук рассчитан на запуск в **Google Colab**. Ключи API берутся из «Секретов» Colab
> или переменных окружения — в коде их нет.

In [ ]:
# ЯЧЕЙКА 1 — Установка зависимостей
!pip install requests pandas openpyxl pillow scikit-learn -q
print('Готово')

In [ ]:
# ЯЧЕЙКА 2 — Ключи YandexCloud (берутся из секретов, НЕ хранятся в коде)
import os

# В Google Colab: слева значок ключа (Секреты) -> добавь YANDEX_API_KEY и YANDEX_FOLDER_ID
try:
    from google.colab import userdata
    API_KEY   = userdata.get('YANDEX_API_KEY')
    FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')
except Exception:
    API_KEY   = os.environ.get('YANDEX_API_KEY',   'ВСТАВЬ_СВОЙ_КЛЮЧ')
    FOLDER_ID = os.environ.get('YANDEX_FOLDER_ID', 'ВСТАВЬ_FOLDER_ID')

assert API_KEY and not API_KEY.startswith('ВСТАВЬ'), 'Задай YANDEX_API_KEY в секретах Colab'
print('Ключи настроены, folder_id:', FOLDER_ID)

In [ ]:
# ЯЧЕЙКА 3 — База требований ИБ
# ВНИМАНИЕ: реальная база требований является внутренним корпоративным документом
# и в репозиторий НЕ включена. Ниже — вымышленные обобщённые примеры, показывающие
# только ФОРМАТ данных (код, раздел, текст, применимость по классам высокий/средний/низкий).
# Обозначения применимости: 'О' — обязательно, 'Ж' — желательно, 'Б' — не применимо.
import pandas as pd

REQUIREMENTS_DB = [
    ('A.1', 'Аутентификация',        'Все внешние соединения должны использовать протокол TLS версии 1.2 и выше', 'О', 'О', 'О'),
    ('A.2', 'Журналирование и аудит', 'Система должна вести журнал событий безопасности и хранить его установленный срок',  'О', 'О', 'Ж'),
    ('A.3', 'Управление доступом',    'Доступ к функциям системы предоставляется по принципу минимальных привилегий',       'О', 'О', 'О'),
    ('A.4', 'Защита данных',          'Персональные данные должны храниться в зашифрованном виде',                            'О', 'Ж', 'Б'),
    # ... остальные требования (из корпоративного документа) сюда не включаются ...
]

df_req = pd.DataFrame(REQUIREMENTS_DB, columns=['code','section','requirement','высокий','средний','низкий'])
print('База загружена:', len(df_req), 'требований (пример формата)')
df_req.head()

In [ ]:
# ЯЧЕЙКА 4 — Функции агента (YandexCloud: Vision OCR + YandexGPT)
VISION_URL = 'https://ocr.api.cloud.yandex.net/ocr/v1/recognizeText'
GPT_URL    = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'

def get_headers():
    return {'Authorization': 'Api-Key ' + API_KEY, 'Content-Type': 'application/json'}

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def yandex_vision(image_path):
    img_b64 = encode_image(image_path)
    mime = 'image/png' if image_path.lower().endswith('.png') else 'image/jpeg'
    payload = {'mimeType': mime, 'languageCodes': ['ru','en'], 'content': img_b64}
    resp = requests.post(VISION_URL, headers=get_headers(), json=payload)
    resp.raise_for_status()
    texts = []
    for block in resp.json().get('result',{}).get('textAnnotation',{}).get('blocks',[]):
        for line in block.get('lines',[]):
            t = ' '.join(w.get('text','') for w in line.get('words',[]))
            if t.strip(): texts.append(t.strip())
    return '\n'.join(texts)

def yandex_gpt(prompt, max_tokens=2000):
    payload = {
        'modelUri': 'gpt://' + FOLDER_ID + '/yandexgpt/latest',
        'completionOptions': {'stream': False, 'temperature': 0.3, 'maxTokens': max_tokens},
        'messages': [{'role': 'user', 'text': prompt}]
    }
    resp = requests.post(GPT_URL, headers=get_headers(), json=payload)
    resp.raise_for_status()
    return resp.json()['result']['alternatives'][0]['message']['text']

def step1_analyze_architecture(image_path, system_name, criticality, has_pdn):
    print('Шаг 1: Читаю схему через Yandex Vision...')
    ocr_text = yandex_vision(image_path)
    print('  Извлечено символов:', len(ocr_text))
    pdn_str = 'да, обрабатываются ПДн' if has_pdn else 'нет'
    examples_block = ''
    if FEW_SHOT_EXAMPLES:
        examples_block = '\n\nПРИМЕРЫ ПРАВИЛЬНОГО АНАЛИЗА:\n'
        for i, ex in enumerate(FEW_SHOT_EXAMPLES):
            examples_block += ('\nПример ' + str(i+1) + ':\n'
                'Текст со схемы: ' + ex['ocr_text'][:400] + '...\n'
                'Правильные требования:\n' + ex['requirements'] + '\n')
        examples_block += '\nПо аналогии проанализируй новую схему.\n'
        print('  Используется примеров:', len(FEW_SHOT_EXAMPLES))
    prompt = (
        'Ты эксперт по ИБ. Система: ' + system_name + ', класс: ' + criticality + ', ПДн: ' + pdn_str + '\n'
        'Если встречаешь HTTPS, OAuth, token, JWT, KeyCloak, PostgreSQL, API — включи их.\n'
        + examples_block +
        'Текст со схемы:\n' + ocr_text + '\n'
        'Верни ТОЛЬКО JSON без markdown:\n'
        '{"components":[{"name":"...","type":"...","description":"..."}],'
        '"data_flows":[{"from":"...","to":"...","data":"...","protocol":"...","flow_num":"..."}],'
        '"has_public_access":true,"has_api":true,"has_db":true,"has_iam":true,'
        '"has_microservices":true,"has_external_ai":true,"has_edoc":false,'
        '"security_observations":["..."]}'
    )
    raw = ask_llm(prompt, 2000).strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    raw = raw.strip().rstrip('`')
    result = json.loads(raw)
    print('  Компонентов:', len(result.get('components',[])), '  Потоков:', len(result.get('data_flows',[])))
    return result

def step3_generate_report(system_name, criticality, has_pdn, arch, reqs):
    print('\nШаг 3: Генерирую отчёт...')
    obs  = '\n'.join('- ' + o for o in arch.get('security_observations',[]))
    comp = '\n'.join('- ' + c['name'] + ' (' + c['type'] + '): ' + c['description'] for c in arch.get('components',[]))
    top  = '\n'.join('[' + r['code'] + '] ' + r['requirement'][:80] + ' (' + r['label'] + ')' for r in reqs[:15])
    pdn_str = 'Да' if has_pdn else 'Нет'
    prompt = ('Эксперт по ИБ. Отчёт для ' + system_name + ' (' + criticality + ', ПДн: ' + pdn_str + ').\n'
        'Компоненты:\n' + comp + '\nНаблюдения:\n' + obs + '\nТребования:\n' + top + '\n\n'
        'Разделы: 1.Резюме системы 2.Ключевые риски 3.Приоритеты выполнения 4.Особые замечания')
    return ask_llm(prompt, 1500)

import base64, json, requests
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

FEW_SHOT_EXAMPLES = []
reqs_before_edit  = []
print('Базовые функции агента готовы!')

## Версия для Сбер GPT (GigaChat)

Логика агента общая — различается только обращение к модели. Ниже определены функции
для GigaChat, а переключатель `MODEL` выбирает, какая модель используется:
`'yandex'` (YandexGPT) или `'gigachat'` (Сбер GPT). Так можно запустить один и тот же
конвейер на обеих моделях и сравнить метрики.

In [ ]:
# ЯЧЕЙКА 4г — Функции для Сбер GPT (GigaChat) + переключатель модели
import uuid
import urllib3
urllib3.disable_warnings()  # GigaChat использует сертификат НУЦ Минцифры

# Ключ авторизации GigaChat берётся из секретов (см. .env.example)
try:
    from google.colab import userdata
    GIGACHAT_AUTH_KEY = userdata.get('GIGACHAT_AUTH_KEY')
except Exception:
    GIGACHAT_AUTH_KEY = os.environ.get('GIGACHAT_AUTH_KEY', 'ВСТАВЬ_AUTH_KEY')

GIGACHAT_OAUTH_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth'
GIGACHAT_API_URL   = 'https://gigachat.devices.sberbank.ru/api/v1/chat/completions'
_giga_token = None

def get_gigachat_token():
    # обмениваем Authorization Key на токен доступа (действует 30 минут)
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json',
        'RqUID': str(uuid.uuid4()),
        'Authorization': 'Basic ' + GIGACHAT_AUTH_KEY,
    }
    data = {'scope': 'GIGACHAT_API_PERS'}  # для юрлиц: 'GIGACHAT_API_CORP'
    resp = requests.post(GIGACHAT_OAUTH_URL, headers=headers, data=data, verify=False)
    resp.raise_for_status()
    return resp.json()['access_token']

def gigachat(prompt, max_tokens=2000):
    global _giga_token
    if _giga_token is None:
        _giga_token = get_gigachat_token()
    headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + _giga_token}
    payload = {'model': 'GigaChat',
               'messages': [{'role': 'user', 'content': prompt}],
               'temperature': 0.3, 'max_tokens': max_tokens}
    resp = requests.post(GIGACHAT_API_URL, headers=headers, json=payload, verify=False)
    if resp.status_code == 401:            # токен истёк — обновляем и повторяем
        _giga_token = get_gigachat_token()
        headers['Authorization'] = 'Bearer ' + _giga_token
        resp = requests.post(GIGACHAT_API_URL, headers=headers, json=payload, verify=False)
    resp.raise_for_status()
    return resp.json()['choices'][0]['message']['content']

# ── Переключатель модели ────────────────────────────────
MODEL = 'yandex'   # 'yandex' — YandexGPT, 'gigachat' — Сбер GPT

def ask_llm(prompt, max_tokens=2000):
    if MODEL == 'gigachat':
        return gigachat(prompt, max_tokens)
    return yandex_gpt(prompt, max_tokens)

print('Функции GigaChat готовы. Текущая модель:', MODEL)

## Шаг 2 — трёхэтапный отбор требований

1. **Фильтрация по классу критичности** — убираем неприменимые (`Б`).
2. **Контекстная фильтрация** по архитектурным флагам (есть ли API, БД, IAM, ПДн, веб и т.д.).
3. **Ранжирование по TF-IDF** — оставляем топ-N наиболее релевантных архитектуре требований.

In [ ]:
# ЯЧЕЙКА 4а — ШАГ 2: трёхэтапный отбор требований
TOP_N_REQUIREMENTS = 50  # максимальное кол-во итоговых требований

def step2_map_requirements(arch, criticality, has_pdn):
    print('\nШаг 2: Трёхэтапный отбор требований...')

    # ── ЭТАП 1: фильтрация по классу критичности ────────
    after_stage1 = []
    for _, row in df_req.iterrows():
        appl = row[criticality]
        if appl == 'Б':
            continue
        after_stage1.append({
            'code': row['code'], 'section': row['section'],
            'requirement': row['requirement'],
            'applicability': appl,
            'label': 'Обязательно' if appl == 'О' else 'Желательно'
        })
    print('  Этап 1 (фильтр по классу):', len(after_stage1), 'требований')

    # ── ЭТАП 2: контекстная фильтрация по архитектуре ───
    has_iam    = arch.get('has_iam', False)
    has_api    = arch.get('has_api', False)
    has_db     = arch.get('has_db', False)
    has_micro  = arch.get('has_microservices', False)
    has_public = arch.get('has_public_access', False)
    has_ext_ai = arch.get('has_external_ai', False)
    has_web    = has_public or has_api
    has_edoc   = arch.get('has_edoc', False)

    after_stage2 = []
    for r in after_stage1:
        section = r['section']; req_lower = r['requirement'].lower()
        include = False; trigger = 'всегда'
        if section in ['Аутентификация', 'Авторизация', 'Управление доступом',
                       'Управление секретами', 'Управление сессией', 'Журналирование и аудит',
                       'Инфраструктурные требования', 'Защита сети', 'Интеграция с системами ИБ']:
            include = True; trigger = 'всегда'
        elif section == 'Криптография при работе с электронными документами':
            if has_edoc: include = True; trigger = 'электронная подпись / ЭП / ЭЦП'
        elif section == 'Шифрование данных':
            if has_public or has_api or has_ext_ai: include = True; trigger = 'внешние соединения / API'
        elif section == 'Защита данных':
            if has_pdn: include = True; trigger = 'ПДн в системе'
        elif section == 'Безопасность Web':
            if has_web: include = True; trigger = 'веб-интерфейс / публичный доступ'
        elif section == 'Криптография при аутентификации/идентификации':
            if has_iam: include = True; trigger = 'IAM / OAuth / KeyCloak / токены'

        if include:
            if any(k in req_lower for k in ['персональн', 'пдн']) and not has_pdn:
                include = False
            elif 'микросервис' in req_lower and not has_micro:
                include = False
            elif 'pam' in req_lower and criticality == 'низкий':
                include = False
            elif any(k in req_lower for k in ['периметров','ddos','dos','waf','ngfw']) and not has_public:
                include = False

        if include:
            r['trigger'] = trigger
            after_stage2.append(r)
    print('  Этап 2 (контекст. фильтр):', len(after_stage2), 'требований')

    # ── ЭТАП 3: ранжирование по TF-IDF сходству ─────────
    arch_description = ' '.join(
        [c.get('name','')+' '+c.get('type','')+' '+c.get('description','') for c in arch.get('components',[])] +
        [f.get('from','')+' '+f.get('to','')+' '+f.get('data','')+' '+f.get('protocol','') for f in arch.get('data_flows',[])] +
        arch.get('security_observations',[]))
    req_texts = [r['requirement'] for r in after_stage2]

    if not req_texts or not arch_description.strip():
        after_stage3 = after_stage2[:TOP_N_REQUIREMENTS]
    else:
        all_texts = [arch_description] + req_texts
        vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)
        vectorizer.fit(all_texts)
        sims = cosine_similarity(vectorizer.transform([arch_description]), vectorizer.transform(req_texts))[0]
        for i, r in enumerate(after_stage2):
            r['relevance'] = float(sims[i])
        mandatory = sorted([r for r in after_stage2 if r['applicability']=='О'], key=lambda x:-x['relevance'])
        desired   = sorted([r for r in after_stage2 if r['applicability']=='Ж'], key=lambda x:-x['relevance'])
        after_stage3 = (mandatory + desired)[:TOP_N_REQUIREMENTS]

    print('  Этап 3 (топ-'+str(TOP_N_REQUIREMENTS)+' по релевантности):', len(after_stage3), 'требований')
    print('  Обязательных:', sum(1 for r in after_stage3 if r['applicability']=='О'),
          '  Желательных:', sum(1 for r in after_stage3 if r['applicability']=='Ж'))
    return after_stage3

In [ ]:
# ЯЧЕЙКА 4б — Вспомогательные функции: метрики, вывод, сохранение
def print_metrics(label, tp, total_a, total_b):
    precision = tp / total_a * 100 if total_a else 0
    recall    = tp / total_b * 100 if total_b else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
    print('\n' + '='*52); print('  ' + label); print('='*52)
    print('  Требований в наборе A:   ', total_a)
    print('  Требований в наборе B:   ', total_b)
    print('  Совпадений (TP):         ', tp)
    print('  Точность (Precision):    ', round(precision, 1), '%')
    print('  Полнота   (Recall):      ', round(recall,    1), '%')
    print('  F1-мера:                 ', round(f1,        1), '%')
    print('='*52)
    return round(precision,1), round(recall,1), round(f1,1)

def display_results(system_name, criticality, arch, reqs, report_text):
    print('\n' + '='*55 + '\n  КОМПОНЕНТЫ АРХИТЕКТУРЫ\n' + '='*55)
    for c in arch.get('components',[]): print('  * ' + c['name'] + '  [' + c['type'] + ']  ' + c['description'])
    print('\n' + '='*55 + '\n  ТАБЛИЦА ИТЕРАЦИЙ\n' + '='*55)
    flows = arch.get('data_flows',[])
    if flows:
        df_f = pd.DataFrame(flows)
        cols = [c for c in ['from','to','data','protocol','flow_num'] if c in df_f.columns]
        df_f = df_f[cols]; df_f.columns = ['От','Куда','Данные','Протокол','Номер'][:len(cols)]
        display(df_f)
    print('\n' + '='*55 + '\n  ТРЕБОВАНИЯ ИБ (' + criticality + ') — ' + str(len(reqs)) + ' шт.\n' + '='*55)
    has_rel = bool(reqs) and 'relevance' in reqs[0]
    cols_r = ['code','section','requirement','label','trigger'] + (['relevance'] if has_rel else [])
    df_r = pd.DataFrame(reqs)[cols_r]
    df_r.columns = ['Код','Раздел','Требование','Статус','Триггер'] + (['Релевантность'] if has_rel else [])
    display(df_r)
    print('\n' + '='*55 + '\n  ИТОГОВЫЙ ОТЧЁТ\n' + '='*55)
    print(report_text)

def save_to_excel(system_name, arch, reqs):
    fname = 'IB_' + system_name.replace(' ','_') + '.xlsx'
    with pd.ExcelWriter(fname, engine='openpyxl') as w:
        flows = arch.get('data_flows',[])
        if flows:
            df_f = pd.DataFrame(flows)
            cols = [c for c in ['from','to','data','protocol','flow_num'] if c in df_f.columns]
            df_f = df_f[cols]; df_f.columns = ['От','Куда','Данные','Протокол','Номер'][:len(cols)]
            df_f.to_excel(w, sheet_name='Итерации', index=False)
        rename_cols = {'code':'Код','section':'Раздел','requirement':'Требование',
            'applicability':'Применимость','label':'Статус','trigger':'Триггер','relevance':'Релевантность'}
        df_r = pd.DataFrame(reqs); df_r = df_r[[c for c in rename_cols if c in df_r.columns]]
        df_r.rename(columns=rename_cols).to_excel(w, sheet_name='Требования ИБ', index=False)
        pd.DataFrame(arch.get('components',[])).to_excel(w, sheet_name='Компоненты', index=False)
    print('Сохранено:', fname)
    try:
        from google.colab import files; files.download(fname)
    except Exception:
        pass

print('Все функции готовы!')

## Few-shot обучение

Перед основным запуском агенту показывают несколько примеров «схема → правильные требования»,
чтобы повысить точность анализа. Каждый пример — фото схемы + xlsx с эталонными требованиями.

In [ ]:
# ЯЧЕЙКА 4в — Обучение на примерах (few-shot)
def load_training_example():
    from google.colab import files
    import openpyxl
    print('Шаг 1 из 2: Загрузи фото схемы-примера (PNG или JPG):')
    uploaded_img = files.upload()
    img_path = list(uploaded_img.keys())[0]
    print('\nШаг 2 из 2: Загрузи файл с правильными требованиями (xlsx):')
    uploaded_req = files.upload()
    req_path = list(uploaded_req.keys())[0]
    print('Читаю схему...')
    ocr_text = yandex_vision(img_path)
    wb = openpyxl.load_workbook(req_path); ws = wb.active
    expert_reqs = []
    for row in ws.iter_rows(values_only=True):
        if row[0] and row[1] is None and isinstance(row[0], str) and row[0] != 'Код': continue
        if row[0] == 'Код': continue
        if row[0] and isinstance(row[0], str) and row[1]:
            expert_reqs.append('[' + str(row[0]).strip() + '] ' + str(row[1]).strip()[:80])
    FEW_SHOT_EXAMPLES.append({'ocr_text': ocr_text, 'requirements': '\n'.join(expert_reqs[:25])})
    print('Пример добавлен! Всего примеров:', len(FEW_SHOT_EXAMPLES))

# Загрузи 1-3 обучающих примера:
# load_training_example()

## Основной запуск агента

Загружаем архитектурную схему анализируемой системы и задаём её параметры
(`SYSTEM_NAME`, `CRITICALITY`, `HAS_PDN`), после чего агент проходит все три шага.

In [ ]:
# ЯЧЕЙКА 5 — ЗАПУСК
from google.colab import files
print('Выбери файл схемы (PNG или JPG):')
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print('Загружен:', image_path)

SYSTEM_NAME = 'Пример-сервис'
CRITICALITY = 'низкий'   # высокий / средний / низкий
HAS_PDN     = True

arch = step1_analyze_architecture(image_path, SYSTEM_NAME, CRITICALITY, HAS_PDN)
reqs = step2_map_requirements(arch, CRITICALITY, HAS_PDN)
reqs_before_edit = list(reqs)
print('\nПервичные требования сохранены.')
report = step3_generate_report(SYSTEM_NAME, CRITICALITY, HAS_PDN, arch, reqs)
display_results(SYSTEM_NAME, CRITICALITY, arch, reqs, report)
save_to_excel(SYSTEM_NAME, arch, reqs)

In [ ]:
# ЯЧЕЙКА 5б — Вывод потоков для ручной правки (копируется в ячейку 6)
print('Скопируй строки ниже в flows_manual в ячейке 6:\n')
for f in arch.get('data_flows', []):
    print("    {'from': '" + str(f.get('from','')) +
          "', 'to': '" + str(f.get('to','')) +
          "', 'data': '" + str(f.get('data','')) +
          "', 'protocol': '" + str(f.get('protocol','HTTPS')) +
          "', 'flow_num': '" + str(f.get('flow_num','')) + "'},")

## Сравнение с экспертом (оценка качества)

Требования агента сравниваются с эталонными требованиями, составленными экспертом вручную.
Сопоставление — по косинусному сходству TF-IDF-векторов; по числу совпадений считаются
Precision, Recall и F1.

In [ ]:
# ЯЧЕЙКА 5в — Сравнение ДО редактирования vs эксперт
from google.colab import files
import openpyxl
SIMILARITY_THRESHOLD = 0.35

print('Загрузи файл с требованиями эксперта (xlsx):')
uploaded_expert = files.upload()
expert_file = list(uploaded_expert.keys())[0]
wb = openpyxl.load_workbook(expert_file); ws = wb.active
expert_rows = []
for row in ws.iter_rows(values_only=True):
    if row[0] and row[1] is None and isinstance(row[0], str) and row[0] != 'Код': continue
    if row[0] == 'Код': continue
    if row[0] and isinstance(row[0], str) and row[1]:
        expert_rows.append({'code': str(row[0]).strip(),
            'requirement': str(row[1]).strip() if row[1] else '',
            'feasibility': str(row[4]).strip() if row[4] else '',
            'comment':     str(row[5]).strip() if row[5] else ''})
df_expert = pd.DataFrame(expert_rows)
print('Загружено требований эксперта:', len(df_expert))

system_texts = [r['requirement'] for r in reqs_before_edit]
expert_texts = df_expert['requirement'].tolist()
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), min_df=1)
vectorizer.fit(system_texts + expert_texts)
sim_matrix = cosine_similarity(vectorizer.transform(expert_texts), vectorizer.transform(system_texts))

matched_expert = set(); matched_system = set(); mapping = []
for ei in range(len(expert_texts)):
    bi = int(np.argmax(sim_matrix[ei])); bs = float(sim_matrix[ei][bi])
    if bs >= SIMILARITY_THRESHOLD:
        mapping.append((ei, bi, bs)); matched_expert.add(ei); matched_system.add(bi)
tp = len(matched_expert)
precision, recall, f1 = print_metrics(
    'МЕТРИКИ: система ДО редактирования vs эксперт (порог='+str(SIMILARITY_THRESHOLD)+')',
    tp, len(reqs_before_edit), len(expert_rows))

## Ручная корректировка потоков и пересчёт

Таблицу потоков данных со схемы можно поправить вручную (OCR не всегда идеален),
после чего архитектурные флаги и список требований пересчитываются автоматически.

In [ ]:
# ЯЧЕЙКА 6 — Ручная правка потоков и пересчёт требований
# (названия компонентов обезличены — приведены как обобщённый пример)
flows_manual = [
    {'from': 'Веб-клиент',   'to': 'Сервис-А',  'data': 'JSON-запрос, AccessToken', 'protocol': 'HTTPS', 'flow_num': '1'},
    {'from': 'Сервис-А',     'to': 'База данных','data': 'Запрос данных',            'protocol': 'TLS',   'flow_num': '2'},
    # ... остальные потоки данных со схемы ...
    {'from': 'IAM-сервис',   'to': 'Веб-клиент', 'data': 'AccessToken',             'protocol': 'HTTPS', 'flow_num': '3'},
]
arch['data_flows'] = flows_manual
df_f = pd.DataFrame(flows_manual)[['from','to','data','protocol','flow_num']]
df_f.columns = ['От','Куда','Данные','Протокол','Номер']
print('Обновлённая таблица итераций:'); display(df_f)

all_text = ' '.join(str(f.get('from',''))+' '+str(f.get('to',''))+' '+str(f.get('data','')) for f in flows_manual).lower()
arch['has_iam']           = any(k in all_text for k in ['keycloak','token','oauth','jwt','iam'])
arch['has_api']           = any(k in all_text for k in ['api','rest','http'])
arch['has_db']            = any(k in all_text for k in ['postgresql','mysql','бд','база'])
arch['has_microservices'] = any(k in all_text for k in ['сервис','service','микросервис'])
arch['has_external_ai']   = any(k in all_text for k in ['giga','chat','gpt','llm'])
arch['has_public_access'] = any(k in all_text for k in ['клиент','client','внешн','публичн'])
arch['has_web']           = arch['has_public_access'] or arch['has_api']

reqs = step2_map_requirements(arch, CRITICALITY, HAS_PDN)
df_r = pd.DataFrame(reqs)[['code','section','requirement','label','trigger']]
df_r.columns = ['Код','Раздел','Требование','Статус','Триггер']
print('\nПересчитанные требования —', len(reqs), 'шт.:'); display(df_r)

In [ ]:
# ЯЧЕЙКА 7 — Сохранить итоговый Excel
save_to_excel(SYSTEM_NAME, arch, reqs)

In [ ]:
# ЯЧЕЙКА 8 — Сравнение ДО и ПОСЛЕ ручного редактирования
codes_before = set(r['code'] for r in reqs_before_edit)
codes_after  = set(r['code'] for r in reqs)
added   = codes_after  - codes_before
removed = codes_before - codes_after
same    = codes_before & codes_after

print_metrics('МЕТРИКИ: ДО vs ПОСЛЕ редактирования таблицы', len(same), len(codes_after), len(codes_before))
print('\n  Precision — доля требований ПОСЛЕ, которые были и ДО')
print('  Recall    — доля требований ДО, которые сохранились ПОСЛЕ')

if added:
    print('\nДОБАВЛЕНЫ:')
    for code_ in sorted(added):
        r = next(x for x in reqs if x['code'] == code_); print('  + [' + code_ + '] ' + r['requirement'][:70])
if removed:
    print('\nУБРАНЫ:')
    for code_ in sorted(removed):
        r = next(x for x in reqs_before_edit if x['code'] == code_); print('  - [' + code_ + '] ' + r['requirement'][:70])

In [ ]:
# ЯЧЕЙКА 9 — Сравнение ПОСЛЕ редактирования vs эксперт
from google.colab import files
import openpyxl
SIMILARITY_THRESHOLD = 0.35

print('Загрузи файл с требованиями эксперта (xlsx):')
uploaded_expert = files.upload()
expert_file = list(uploaded_expert.keys())[0]
wb = openpyxl.load_workbook(expert_file); ws = wb.active
expert_rows = []
for row in ws.iter_rows(values_only=True):
    if row[0] and row[1] is None and isinstance(row[0], str) and row[0] != 'Код': continue
    if row[0] == 'Код': continue
    if row[0] and isinstance(row[0], str) and row[1]:
        expert_rows.append({'code': str(row[0]).strip(), 'requirement': str(row[1]).strip() if row[1] else ''})
df_expert = pd.DataFrame(expert_rows)
print('Загружено требований эксперта:', len(df_expert))

system_texts = [r['requirement'] for r in reqs]
expert_texts = df_expert['requirement'].tolist()
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), min_df=1)
vectorizer.fit(system_texts + expert_texts)
sim_matrix = cosine_similarity(vectorizer.transform(expert_texts), vectorizer.transform(system_texts))

matched_expert = set()
for ei in range(len(expert_texts)):
    if float(np.max(sim_matrix[ei])) >= SIMILARITY_THRESHOLD:
        matched_expert.add(ei)
tp = len(matched_expert)
precision, recall, f1 = print_metrics(
    'МЕТРИКИ: система ПОСЛЕ редактирования vs эксперт (порог='+str(SIMILARITY_THRESHOLD)+')',
    tp, len(reqs), len(expert_rows))